# 位置编码（Positional Encoding）

Transformer 会一次性读入所有词，
本身不知道哪个词在前、哪个在后。
但词序很重要——
「The dog bit the man」和「The man bit the dog」含义完全不同。

因此需要在送入模型前，给每个词打上位置标记。
现代做法是旋转位置嵌入（Rotary Position Embeddings，简称 RoPE）。
RoPE 不是直接加位置数字，而是按词在句中的位置旋转向量。

巧妙之处在于：旋转后，任意两个词的点积
只取决于它们相距多远，而非绝对位置。
相隔 5 个位置的词，无论在第 0 位还是第 500 位，
相对角度都相同——这正是注意力机制所关心的。

本 notebook 展示旋转的数学原理，
并演示向量在不同位置如何变化。

## 导入

In [ ]:
import torch
import torch.nn as nn
import math

## RoPE 实现

RoPE 期望输入张量形状为 [batch, num_heads, seq_len, head_dim]。
这与多头注意力内部使用的布局相同。

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even"

        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)

        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)

## 观察单个向量的旋转

In [ ]:
rope = RotaryPositionalEmbedding(d_model=4, max_seq_len=16)

q = torch.tensor([[[[0.8, 0.3, -0.5, 0.2]]]])  # [1, 1, 1, 4]
print(f"原始向量: {q[0, 0, 0].tolist()}")
print()

for pos in [0, 1, 2, 5]:
    rotated = rope(q, seq_len=pos + 1)
    last = rotated[0, 0, -1].tolist()
    vals = [f"{v:.3f}" for v in last]
    print(f"位置 {pos}: [{', '.join(vals)}]")

## 相对位置性质

两个旋转后向量的点积只取决于它们之间的距离，
与绝对位置无关。

In [ ]:
rope = RotaryPositionalEmbedding(d_model=64, max_seq_len=256)

seq_len = 12
num_heads = 4
head_dim = 64

q = torch.randn(1, num_heads, seq_len, head_dim)
k = torch.randn(1, num_heads, seq_len, head_dim)

q_rot = rope(q, seq_len=seq_len)
k_rot = rope(k, seq_len=seq_len)

head = 0
a = (q_rot[0, head, 2] @ k_rot[0, head, 4]).item()
b = (q_rot[0, head, 5] @ k_rot[0, head, 7]).item()
c = (q_rot[0, head, 0] @ k_rot[0, head, 8]).item()

print(f"距离 2（位置 2,4）:     {a:.4f}")
print(f"距离 2（位置 5,7）:     {b:.4f}")
print(f"距离 8（位置 0,8）:     {c:.4f}")
print()
print("相同距离会得到相近的分数，与绝对位置无关。")

## 注意力中的 RoPE

RoPE 只作用于 query 和 key，value 保持不变。
这样注意力分数能感知位置，同时保持内容信息清晰。

In [ ]:
head_dim = 64
num_heads = 4
seq_len = 10

rope = RotaryPositionalEmbedding(d_model=head_dim, max_seq_len=128)

q = torch.randn(1, num_heads, seq_len, head_dim)
k = torch.randn(1, num_heads, seq_len, head_dim)

q_rot = rope(q, seq_len)
k_rot = rope(k, seq_len)

scores = (q_rot @ k_rot.transpose(-2, -1)) / math.sqrt(head_dim)
print(f"注意力分数形状: {scores.shape}")
print(f"Token 0 关注自身:  {scores[0, 0, 0, 0]:.4f}")
print(f"Token 0 关注 token 5: {scores[0, 0, 0, 5]:.4f}")